# Datainsamling

Datainsamling av de kurser jag valt ut att anända mig utav.

In [29]:
# Kurserna och dess ID:n som jag valt att ta med.

COURSES = {
    3064200: "Segling för nybörjare",
    1823163: "Säkerhet ombord",
    1703530: "Ansvariga vem ska jag kontakta?",
    629026: "Destination Kroatien",
    629032: "Mathantering",
    629027: "Båthantering",
    643236: "Våra båtar",
    631751: "Genomförande av seglingsresa",
    631725: "Riktlinjer",
    629034: "Kundnöjdhet",
    628420: "More Sailing Vilka är vi?"
}

In [ ]:
import os
import json
import requests
import time
from pathlib import Path
from dotenv import load_dotenv

In [31]:
import os
import json
import requests
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("THINKIFIC_API_ACCESS_TOKEN")
SUBDOMAIN = os.getenv("THINKIFIC_SUBDOMAIN")
URL = "https://api.thinkific.com/stable/graphql"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "X-Thinkific-Subdomain": SUBDOMAIN
}

course_query_template = """
query GetCourseStructure {{
  course(id: {course_id}) {{
    id
    name
    slug
    curriculum {{
      chaptersCount
      lessonsCount
      chapters(first: 50) {{
        nodes {{
          id
          title
          position
          lessons(first: 100) {{
            nodes {{
              id
              title
              lessonType
            }}
          }}
        }}
      }}
    }}
  }}
}}
"""

lesson_query_template = """
query GetLessonText {{
  lesson(id: {lesson_id}) {{
    id
    title
    lessonType
    takeUrl
    chapter {{
      id
      title
    }}
    course {{
      id
      name
      slug
    }}
    content {{
      __typename
      ... on TextContent {{
        id
        contentType
        htmlDescription
      }}
    }}
  }}
}}
"""

def run_query(query: str, max_retries: int = 5) -> dict:
    for attempt in range(max_retries):
        response = requests.post(
            URL,
            headers=headers,
            json={"query": query},
            timeout=30
        )

        # Om rate limited
        if response.status_code == 429:
            wait_time = 5 * (attempt + 1)
            print(f"429 Too Many Requests. Väntar {wait_time} sekunder...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()
        data = response.json()

        # GraphQL kan ibland returnera 200 men ändå ha errors
        if "errors" in data:
            error_text = json.dumps(data["errors"], indent=2, ensure_ascii=False)

            # Om 429 gömmer sig i GraphQL-felet
            if "Too Many Requests" in error_text:
                wait_time = 5 * (attempt + 1)
                print(f"GraphQL rate limit. Väntar {wait_time} sekunder...")
                time.sleep(wait_time)
                continue

            raise ValueError(error_text)

        # liten paus för att vara snäll mot API:t
        time.sleep(0.5)

        return data

    raise ValueError("Misslyckades efter flera försök på grund av rate limiting.")


def export_course(course_id, course_name):
    print(f"\n--- Exporterar: {course_name} ({course_id}) ---")

    raw_dir = Path(f"data/raw/{course_id}")
    raw_dir.mkdir(parents=True, exist_ok=True)

    # Hämta kursstruktur
    course_query = course_query_template.format(course_id=course_id)
    course_data = run_query(course_query)

    course = course_data["data"]["course"]

    if course is None:
        print(f"❌ Kunde inte hämta kurs {course_id}")
        return

    with open(raw_dir / "course_structure.json", "w", encoding="utf-8") as f:
        json.dump(course_data, f, indent=2, ensure_ascii=False)

    print(f"Kurs: {course['name']}")

    # Hämta alla lessons
    for chapter in course["curriculum"]["chapters"]["nodes"]:
        for lesson in chapter["lessons"]["nodes"]:
            lesson_id = lesson["id"]

            lesson_query = lesson_query_template.format(lesson_id=lesson_id)
            lesson_data = run_query(lesson_query)

            lesson_obj = lesson_data["data"]["lesson"]
            content = lesson_obj.get("content") or {}

            output = {
                "course_id": lesson_obj["course"]["id"],
                "course_name": lesson_obj["course"]["name"],
                "course_slug": lesson_obj["course"]["slug"],
                "chapter_id": lesson_obj["chapter"]["id"],
                "chapter_title": lesson_obj["chapter"]["title"],
                "lesson_id": lesson_obj["id"],
                "lesson_title": lesson_obj["title"].strip(),
                "lesson_type": lesson_obj["lessonType"],
                "take_url": lesson_obj["takeUrl"],
                "content_typename": content.get("__typename"),
                "content_id": content.get("id"),
                "content_type": content.get("contentType"),
                "html_description": content.get("htmlDescription")
            }

            with open(raw_dir / f"{lesson_id}.json", "w", encoding="utf-8") as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            print(f"  ✓ Lesson {lesson_id}")

    print(f"✔ Klar: {course_name}")


def main():
    for course_id, course_name in COURSES.items():
        export_course(course_id, course_name)


if __name__ == "__main__":
    main()


--- Exporterar: Segling för nybörjare (3064200) ---
Kurs: Segling för Nybörjare
  ✓ Lesson 73644304
  ✓ Lesson 63949181
  ✓ Lesson 73644264
  ✓ Lesson 63949184
  ✓ Lesson 63949186
✔ Klar: Segling för nybörjare

--- Exporterar: Säkerhet ombord (1823163) ---
Kurs: Säkerhet ombord
  ✓ Lesson 64247538
  ✓ Lesson 64247725
  ✓ Lesson 60038464
  ✓ Lesson 60039047
  ✓ Lesson 60039176
  ✓ Lesson 64248850
  ✓ Lesson 64248666
  ✓ Lesson 64250549
  ✓ Lesson 64249035
  ✓ Lesson 64250560
  ✓ Lesson 64250565
  ✓ Lesson 64250610
  ✓ Lesson 64249271
  ✓ Lesson 72851249
  ✓ Lesson 64248713
  ✓ Lesson 64248748
  ✓ Lesson 64248784
  ✓ Lesson 64248946
  ✓ Lesson 64251035
  ✓ Lesson 64251056
  ✓ Lesson 64249129
  ✓ Lesson 64250757
✔ Klar: Säkerhet ombord

--- Exporterar: Ansvariga vem ska jag kontakta? (1703530) ---
Kurs: Ansvariga- Vem ska jag kontakta?
  ✓ Lesson 31459054
  ✓ Lesson 31459057
  ✓ Lesson 31459076
  ✓ Lesson 31459077
  ✓ Lesson 31460254
  ✓ Lesson 31460310
  ✓ Lesson 31460350
  ✓ Lesson 314

CLeaning

In [32]:
import json
from pathlib import Path
from bs4 import BeautifulSoup

COURSES = [
    3064200, 1823163, 1703530, 629026, 629032,
    629027, 643236, 631751, 631725, 629034, 628420
]

for course_id in COURSES:
    print(f"\n--- Cleaning course {course_id} ---")

    raw_dir = Path(f"data/raw/{course_id}")
    cleaned_dir = Path(f"data/cleaned/{course_id}")
    cleaned_dir.mkdir(parents=True, exist_ok=True)

    for file_path in raw_dir.glob("*.json"):
        if file_path.name == "course_structure.json":
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        html = data.get("html_description") or ""
        text = BeautifulSoup(html, "html.parser").get_text("\n", strip=True)

        cleaned = {
            "course_id": data["course_id"],
            "course_name": data["course_name"],
            "chapter_id": data["chapter_id"],
            "chapter_title": data["chapter_title"],
            "lesson_id": data["lesson_id"],
            "lesson_title": data["lesson_title"],
            "text": text
        }

        with open(cleaned_dir / file_path.name, "w", encoding="utf-8") as f:
            json.dump(cleaned, f, indent=2, ensure_ascii=False)

        print(f"  ✓ {file_path.name}")

print("\n✔ All cleaning klar")


--- Cleaning course 3064200 ---
  ✓ 63949181.json
  ✓ 63949184.json
  ✓ 63949186.json
  ✓ 73644264.json
  ✓ 73644304.json

--- Cleaning course 1823163 ---
  ✓ 60038464.json
  ✓ 60039047.json
  ✓ 60039176.json
  ✓ 64247538.json
  ✓ 64247725.json
  ✓ 64248666.json
  ✓ 64248713.json
  ✓ 64248748.json
  ✓ 64248784.json
  ✓ 64248850.json
  ✓ 64248946.json
  ✓ 64249035.json
  ✓ 64249129.json
  ✓ 64249271.json
  ✓ 64250549.json
  ✓ 64250560.json
  ✓ 64250565.json
  ✓ 64250610.json
  ✓ 64250757.json
  ✓ 64251035.json
  ✓ 64251056.json
  ✓ 72851249.json

--- Cleaning course 1703530 ---
  ✓ 31459054.json
  ✓ 31459057.json
  ✓ 31459076.json
  ✓ 31459077.json
  ✓ 31460254.json
  ✓ 31460310.json
  ✓ 31460350.json
  ✓ 31460404.json

--- Cleaning course 629026 ---
  ✓ 11088172.json
  ✓ 11088313.json
  ✓ 11088318.json
  ✓ 11088322.json
  ✓ 11088343.json
  ✓ 11088714.json
  ✓ 11088939.json
  ✓ 11088941.json
  ✓ 11088961.json
  ✓ 11088978.json
  ✓ 11088998.json
  ✓ 11128337.json
  ✓ 11297278.json
  ✓ 1